In [1]:
# ========imports=========
import sys
from pathlib import Path

root_path = Path.cwd().parent if Path.cwd().name in ["notebooks"] else Path.cwd()

if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

import pandas as pd
from src.client import complete_gap


In [2]:
test_sentences = [
    {"id": 1, "text": "Battery lasts all day, best purchase I made this year!", "ground_truth": "Positive"},
    {"id": 2, "text": "Customer service was utterly useless and rude.", "ground_truth": "Negative"},
    {"id": 3, "text": "The cinematography was breathtaking, but the pacing felt a bit dragged.", "ground_truth": "Negative"}, # مرزی
    {"id": 4, "text": "Not bad for the price, does the job well enough.", "ground_truth": "Positive"} # مرزی
]

demos_pos = [
    "Review: Outstanding build quality and sleek design.\nSentiment: Positive",
    "Review: Exceeded all my expectations, absolutely love it!\nSentiment: Positive",
    "Review: Fast delivery and works like a charm.\nSentiment: Positive",
    "Review: A masterpiece of modern cinema with brilliant acting.\nSentiment: Positive",
    "Review: Highly recommended for anyone looking for reliability.\nSentiment: Positive"
]

demos_neg = [
    "Review: Broke down after just two days of light use.\nSentiment: Negative",
    "Review: Complete waste of money, do not buy this product.\nSentiment: Negative",
    "Review: Terrible sound quality and painful to wear.\nSentiment: Negative"
]


In [3]:
instruction = "Classify the sentiment of the given review as either 'Positive' or 'Negative'. Return ONLY the single label word (Positive or Negative).\n\n"

def build_prompt(examples_list, query_text):
    examples_str = "\n\n".join(examples_list)
    if examples_str:
        return f"{instruction}{examples_str}\n\nReview: {query_text}\nSentiment:"
    return f"{instruction}Review: {query_text}\nSentiment:"


prompts_zero = []

balanced_examples = [
    demos_pos[0], demos_neg[0],
    demos_pos[1], demos_neg[1],
    demos_pos[2], demos_neg[2]
]

imbalanced_examples = [
    demos_pos[0], demos_pos[1], demos_neg[0], demos_pos[2], demos_pos[3], demos_pos[4]
]

imbalanced_reversed_examples = list(reversed(imbalanced_examples))


In [4]:
def run_experiment(examples, experiment_name):
    results = []
    for item in test_sentences:
        prompt = build_prompt(examples, item["text"])
        res = complete_gap(prompt=prompt, temperature=0)
        prediction = res.choices[0].message.content.strip()

        results.append({
            "id": item["id"],
            "text": item["text"],
            "ground_truth": item["ground_truth"],
            f"{experiment_name}": prediction
        })
    return results


In [5]:
res_zero = run_experiment([], "Zero-shot")
res_balanced = run_experiment(balanced_examples, "Balanced (3P-3N)")
res_imbalanced = run_experiment(imbalanced_examples, "Imbalanced (5P-1N)")
res_reversed = run_experiment(imbalanced_reversed_examples, "Imbalanced_Reversed")

df = pd.DataFrame(test_sentences)
df["Zero-shot"] = [r["Zero-shot"] for r in res_zero]
df["Balanced"] = [r["Balanced (3P-3N)"] for r in res_balanced]
df["Imbalanced"] = [r["Imbalanced (5P-1N)"] for r in res_imbalanced]
df["Imbalanced_Reversed"] = [r["Imbalanced_Reversed"] for r in res_reversed]

df[["id", "ground_truth", "Zero-shot", "Balanced", "Imbalanced", "Imbalanced_Reversed", "text"]]

df.to_csv(root_path / "report" /"ImgPlt"/"few_shot.csv",index=False )

display(df)

,id,text,ground_truth,Zero-shot,Balanced,Imbalanced,Imbalanced_Reversed
0,1,"Battery lasts all day, best purchase I made th...",Positive,Positive,Positive,Positive,Positive
1,2,Customer service was utterly useless and rude.,Negative,Negative,Negative,Negative,Negative
2,3,"The cinematography was breathtaking, but the p...",Negative,Negative,Negative,Negative,Negative
3,4,"Not bad for the price, does the job well enough.",Positive,Positive,Positive,Positive,Positive


In [6]:
test_sentences_hard_test = [
    {    "id": 1,
        "text": "Oh wonderful, another software update that broke every single feature I actually use.",
        "ground_truth": "Negative"},

    {   "id": 2,
        "text": "The package arrived at 3 PM on Tuesday and matched the dimensions on the box.",
        "ground_truth": "Neutral"},

    {   "id": 3,
        "text": "The design is sleek and premium, though it frequently overheats and crashes under normal load.",
        "ground_truth": "Negative"},

    {   "id": 4,
        "text": "It simply exists. It does what it is supposed to do, nothing more and nothing less.",
        "ground_truth": "Neutral"}
]


In [7]:
res_zero = run_experiment([], "Zero-shot")
res_balanced = run_experiment(balanced_examples, "Balanced (3P-3N)")
res_imbalanced = run_experiment(imbalanced_examples, "Imbalanced (5P-1N)")
res_reversed = run_experiment(imbalanced_reversed_examples, "Imbalanced_Reversed")

df = pd.DataFrame(test_sentences_hard_test)
df["Zero-shot"] = [r["Zero-shot"] for r in res_zero]
df["Balanced"] = [r["Balanced (3P-3N)"] for r in res_balanced]
df["Imbalanced"] = [r["Imbalanced (5P-1N)"] for r in res_imbalanced]
df["Imbalanced_Reversed"] = [r["Imbalanced_Reversed"] for r in res_reversed]

df[["id", "ground_truth", "Zero-shot", "Balanced", "Imbalanced", "Imbalanced_Reversed", "text"]]

df.to_csv(root_path / "report" /"ImgPlt"/"few_shot_hard_test.csv",index=False )

display(df)

,id,text,ground_truth,Zero-shot,Balanced,Imbalanced,Imbalanced_Reversed
0,1,"Oh wonderful, another software update that bro...",Negative,Positive,Positive,Positive,Positive
1,2,The package arrived at 3 PM on Tuesday and mat...,Neutral,Negative,Negative,Negative,Negative
2,3,"The design is sleek and premium, though it fre...",Negative,Negative,Negative,Negative,Negative
3,4,It simply exists. It does what it is supposed ...,Neutral,Positive,Positive,Positive,Positive
